In [1]:
!pip install langchain-community pypdf
!pip install langchain-huggingface
!pip install sentence-transformers
!pip install -qU langchain-chroma


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

c:\Users\Técnico em IA\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
"""Load a PDF file and print the number of documents loaded."""

file_path = "D:\\UC15\\senac_ia_uc15_nlp_project\\data\\marcelo\\sindilojas_2025_2026.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

38


In [4]:
"""Split the loaded documents into smaller chunks and print the total number of chunks created."""

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400, chunk_overlap=30, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

print(len(all_splits))

328


In [5]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15326.79it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 768

[0.003042803378775716, -0.03260025754570961, -0.01823962666094303, 0.038968317210674286, 0.031141849234700203, -0.059096015989780426, 0.0029536141082644463, 0.004146444611251354, -0.05678118020296097, -0.02073892205953598]


In [7]:
vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

In [8]:
ids = vector_store.add_documents(documents=all_splits)
ids

['29af61fa-9108-4fce-adfe-0909cde9f3f3',
 '6318f1d7-9230-4328-8da8-6b6616f65844',
 '6e950345-eea0-4df2-a8f3-b3eaf6642b71',
 '8751434e-6db2-4632-a57c-eb4f9eadbc2c',
 '6bc83a5e-5a6b-4821-80ee-9f99239b4728',
 '405a84ad-0166-4990-bdee-d14b06653677',
 'ebfb542b-ae1b-4ca0-97c5-58adaabf1f0d',
 '819169d9-aa47-4b42-8a1a-5799e3f68f48',
 '34d83082-dff0-4061-8258-3ef5f0fdbccc',
 'f5fb0818-2f1f-498d-9eda-9fcecc4b2f3e',
 '921b6e00-750a-4f34-8868-8ec68ec2cfc8',
 'd5aac99b-edfd-4b91-bd0c-42b019a4757d',
 'dcbe19a0-2353-4fc5-ac8a-0d15d155ca8d',
 'ccaafbf1-3708-4b1b-8bb0-9539f6b3e8bf',
 '421087bf-6fb0-464b-88ff-6e53319b26b5',
 'e1f3d4ce-6f66-4484-9ca5-42e7524f0320',
 '14ef559c-8987-4353-8e85-bcbad88bcce9',
 '87e200b3-ce24-4e23-b267-56555ba43ef9',
 'a73356d9-fa78-4dc8-a122-d52a48f12664',
 '0ec71688-dc49-4761-acf8-2cf4295a0b2f',
 '39dc15ed-01b6-41e4-9af4-da497b94cd84',
 '80398f92-716f-47c7-8a9d-de397d403363',
 '2d3c7d9f-9601-45d2-adcb-bb2285c5a7a4',
 '43afe55f-bb4f-4f1f-8170-acb1aa28c8e2',
 '9786274a-0257-

In [10]:
results = vector_store.similarity_search_with_score(
    "Quais são as condições de remuneração e jornada para o trabalho em feriados, conforme a cláusula 41?"
)

doc, score = results[0]
print(f"Score: {score}\n")
print(doc)

Score: 0.561332106590271

page_content='aplicáveis, fica autorizado o trabalho aos feriados: com exceção de  25 de dezembro (Natal) e 
1º de janeiro (Confraternização Universal), desde que atendidas as seguintes regras: 
 
a) comunicação da empresa ao sindicato patronal, da intenção de funci onamento e trabalho 
no mesmo e declaração de que está sendo cumprida integralmente a Convenção Coletiva de 
Trabalho, sendo este documento o indispensável comprovante da regularidade do trabalho; 
 
b) manifestação de vontade por escrito, por parte do empregado, as sistido o menor por seu 
representante legal, em instrumento individual ou plúrimo, do qual conste: 
 
I   – os feriados a serem trabalhados; e 
II  – a discriminação da jornada a ser desenvolvida em cada um. 
 
c) pagamento em dobro das horas efetivamente trabalhadas no feriado, sem prejuízo do 
DSR. Para os comissionistas puros, o cálculo dessa remuneração c orresponderá ao valor de 
mais 1 (um) descanso semanal remunerado, ficando ve